# 작은 NumPy 선형대수 파이프라인 점검

- 기준 TIL: [2026-08-20](../../til/2026/08/2026-08-20.md)
- 관련 강의자료: [01-01](../../materials/private/kant-basic-math/01-01_벡터의_정의와_기하학적_해석.md), [01-02](../../materials/private/kant-basic-math/01-02_내적과_코사인_유사도.md), [01-03](../../materials/private/kant-basic-math/01-03_행렬_연산과_딥러닝_레이어.md), [01-04](../../materials/private/kant-basic-math/01-04_특수_행렬과_행렬_연산_성질.md), [02-01](../../materials/private/kant-basic-math/02-01_선형_변환의_기하학적_해석.md), [02-02](../../materials/private/kant-basic-math/02-02_벡터공간과_선형_독립.md), [02-03](../../materials/private/kant-basic-math/02-03_연립선형방정식과_행렬_해법.md), [03-01](../../materials/private/kant-basic-math/03-01_고유값과_고유벡터.md), [03-02](../../materials/private/kant-basic-math/03-02_행렬_대각화와_PCA_구현.md), [03-03](../../materials/private/kant-basic-math/03-03_직교성과_최소제곱법.md)
- 강의 제공 실습: [01-01](../../materials/private/kant-basic-math/course-provided-practice/01-01_벡터_노름_정규화.md), [01-02](../../materials/private/kant-basic-math/course-provided-practice/01-02_내적과_코사인_유사도.md), [01-03](../../materials/private/kant-basic-math/course-provided-practice/01-03_행렬곱과_완전연결층.md), [01-04](../../materials/private/kant-basic-math/course-provided-practice/01-04_특수_행렬과_행렬_연산_성질.md), [02-01](../../materials/private/kant-basic-math/course-provided-practice/02-01_선형_변환의_기하학적_해석.md), [02-02](../../materials/private/kant-basic-math/course-provided-practice/02-02_벡터공간_선형독립_Rank.md), [02-03](../../materials/private/kant-basic-math/course-provided-practice/02-03_연립방정식과_행렬_해법.md), [03-01](../../materials/private/kant-basic-math/course-provided-practice/03-01_고유값과_고유벡터.md), [03-02](../../materials/private/kant-basic-math/course-provided-practice/03-02_PCA와_고유분해.md), [03-03](../../materials/private/kant-basic-math/course-provided-practice/03-03_직교성과_최소제곱법.md)
- 난이도: Core

작은 입력 벡터를 비교하고, 배치 선형 레이어와 affine 합성을 점검한 뒤, 해의 종류·최소제곱·고유좌표·PCA까지 연결하는 NumPy 도구를 바닥부터 만든다. INDEX에 명시된 강의 제공 실습은 원문 연결만 보존하고 코드나 답은 복사하지 않았다. 이번 TIL에 맞춘 시나리오·테스트·실패 사례를 새로 추가했다.

### 사용 흐름

1. 현재 E번호의 구현 셀에서 함수를 작성한다.
2. 구현 셀을 실행해 최신 정의를 kernel에 반영한다.
3. 바로 아래 fixture 셀로 작은 입력과 중간 상태를 관찰한다.
4. `check_e01()` 형식의 test-check 셀을 실행한다.
5. 관찰한 값과 실패 원인을 `결과 해석`에 자기 말로 적는다.


## Practice Coverage Map

| Outcome ID | TIL location | Practice action | Artifact/Exercise | Required evidence |
| --- | --- | --- | --- | --- |
| O01 | 오늘의 학습 > L2 정규화와 코사인 유사도 | implement | E01 | 정상·직교·영벡터 계약과 feature 단위 변경 뒤 방향 변화를 확인하고 설명 |
| O02 | 오늘의 학습 > 행렬식·condition number·선형 레이어 shape | implement | E02 | determinant와 condition number, batch 출력 shape, 잘못된 shape 거부를 확인 |
| O03 | 오늘의 학습 > affine 변환·열벡터 가중합·합성 순서 | implement | E03 | 열벡터 가중합 trace, 두 단계 출력과 합성 출력 일치, 영입력에서 bias 확인 |
| O04 | 오늘의 학습 > 선형 독립·무한해·최소제곱 투영 | debug | E04 | unique/infinite/inconsistent 분류와 residual 직교 조건을 해석 |
| O05 | 오늘의 학습 > 고유값과 대각화의 좌표 변환 | implement | E05 | 원래 좌표, 고유좌표, 스케일된 좌표, 최종 결과와 음수 고유값의 방향 반전을 기록 |
| O06 | 오늘의 학습 > 중심화·공분산·PCA | implement | E06 | 공분산의 대각·비대각 해석, `N - 1` 공분산의 고유값 정렬, 투영 shape, 0 분산 방향을 확인 |


In [3]:
# setup-check: notebook-only imports
from __future__ import annotations

import numpy as np
import numpy.typing as npt

Array = npt.NDArray[np.float64]


## E01. 안전한 벡터 비교

### 실제 사용 맥락

검색 후보와 질의의 작은 feature 벡터를 비교하기 전에, 길이가 1이 되도록 정규화하고 방향 유사도를 계산한다. 영벡터는 방향이 없으므로 조용히 숫자를 반환하면 안 된다.

### 실행 전 회상·예측

- `[3, 4]`를 정규화한 결과와 그 노름을 적는다.
- `[1, 0]`과 `[0, 7]`의 코사인 유사도를 예측하고, 영벡터 입력에서 어떤 예외가 적절한지 적는다.
- 키를 cm로 쓴 `[170, 70]`과 m로 바꾼 `[1.7, 70]`을 각각 정규화했을 때 방향이 같은지 예측한다.

### 작은 유사 사례와 계약

`l2_normalize([3, 4])`는 `(2,)` float 배열을 반환한다. `cosine_similarity`는 같은 길이의 1D 비영벡터 두 개만 받고 float를 반환한다. 길이가 다르거나 어느 쪽이 영벡터면 `ValueError`다. 한 feature의 단위만 바꾼 뒤 정규화한 두 벡터는 일반적으로 다른 방향이다.

### 구현

바로 아래 구현 셀의 `l2_normalize`와 `cosine_similarity`를 구현한다.

<details>
<summary>힌트 1: 관찰할 상태와 개념</summary>

코사인 유사도의 분모에는 두 L2 노름의 곱이 있다. 어느 한 노름이 0인지 먼저 확인한다.
</details>

<details>
<summary>힌트 2: 작은 trace 또는 Shape 흐름</summary>

`[3, 4]`의 성분 제곱합은 25이고 노름은 5다. 입력 shape `(d,)`는 결과에서도 `(d,)`로 유지한다.
</details>


In [4]:
# TODO: E01
def l2_normalize(vector: Array) -> Array:
    """Return a 1D vector divided by its L2 norm; reject a zero vector."""
    vector = np.asarray(vector, dtype=float)

    if vector.ndim != 1:
        raise ValueError("vector must be 1D")

    if not np.all(np.isfinite(vector)):
        raise ValueError("vector must contain only finite values")

    norm = np.linalg.norm(vector)
    if norm == 0.0:
        raise ValueError("cannot normalize a zero vector")

    return vector / norm


def cosine_similarity(left: Array, right: Array) -> float:
    """Return cosine similarity for equal-shaped, nonzero 1D vectors."""
    left = np.asarray(left, dtype=float)
    right = np.asarray(right, dtype=float)

    if left.shape != right.shape:
        raise ValueError("vectors must have the same shape")

    left_unit = l2_normalize(left)
    right_unit = l2_normalize(right)

    return float(left_unit @ right_unit)


In [5]:
# provided-fixture: E01
# 먼저 E01 구현 셀을 실행하세요.

vector = np.array([3.0, 4.0])
parallel = np.array([6.0, 8.0])
orthogonal_left = np.array([1.0, 0.0])
orthogonal_right = np.array([0.0, 7.0])
centimeters = np.array([170.0, 70.0])
mixed_units = np.array([1.7, 70.0])

normalized_vector = l2_normalize(vector)
parallel_cosine = cosine_similarity(vector, parallel)
orthogonal_cosine = cosine_similarity(orthogonal_left, orthogonal_right)
normalized_centimeters = l2_normalize(centimeters)
normalized_mixed_units = l2_normalize(mixed_units)

print("normalized vector:", normalized_vector)
print("normalized norm:", np.linalg.norm(normalized_vector))
print("parallel cosine:", parallel_cosine)
print("orthogonal cosine:", orthogonal_cosine)
print("centimeters direction:", normalized_centimeters)
print("mixed-units direction:", normalized_mixed_units)
print("same direction:", np.allclose(normalized_centimeters, normalized_mixed_units))


normalized vector: [0.6 0.8]
normalized norm: 1.0
parallel cosine: 1.0
orthogonal cosine: 0.0
centimeters direction: [0.9246781  0.38074981]
mixed-units direction: [0.02427856 0.99970523]
same direction: False


### 테스트와 실패 진단

아래 `check_e01()`로 E01의 정상·경계·실패 계약만 확인한다. 영벡터 검사가 실패하면 분모를 계산하기 전에 노름을 확인했는지 본다.


In [6]:
# test-check: E01
def check_e01() -> None:
    # normal
    np.testing.assert_allclose(
        l2_normalize(np.array([3.0, 4.0])),
        np.array([0.6, 0.8]),
    )
    np.testing.assert_allclose(
        cosine_similarity(np.array([3.0, 4.0]), np.array([6.0, 8.0])),
        1.0,
    )

    # edge
    np.testing.assert_allclose(
        cosine_similarity(np.array([1.0, 0.0]), np.array([0.0, 7.0])),
        0.0,
    )
    centimeters = l2_normalize(np.array([170.0, 70.0]))
    mixed_units = l2_normalize(np.array([1.7, 70.0]))
    np.testing.assert_equal(
        np.allclose(centimeters, mixed_units),
        False,
    )

    # failure
    with np.testing.assert_raises(ValueError):
        l2_normalize(np.array([0.0, 0.0]))
    with np.testing.assert_raises(ValueError):
        cosine_similarity(np.array([1.0, 0.0]), np.array([0.0, 0.0]))
    with np.testing.assert_raises(ValueError):
        cosine_similarity(np.array([1.0, 0.0]), np.array([1.0, 0.0, 0.0]))
    with np.testing.assert_raises(ValueError):
        cosine_similarity(np.ones((1, 2)), np.ones((1, 2)))

    print("E01 checks passed")


check_e01()


E01 checks passed


### 결과 해석

정규화 뒤에 크기가 아니라 방향을 비교한다는 점, 한 feature의 단위 변경이 방향을 바꾸는 이유, 영벡터를 거부하는 것이 왜 수치값을 임의로 만드는 것보다 안전한지 설명한다.


## E02. 배치 선형 레이어와 행렬 상태 점검

### 실제 사용 맥락
작은 NumPy feature 파이프라인에서 한 배치가 레이어를 통과하기 전, 가중치의 조건과 입력·출력 shape을 함께 점검한다.

### 실행 전 회상·예측

- `(2, 3)` batch와 `(2, 3)` weight, `(2,)` bias의 출력 shape을 적는다.
- 대각 원소가 2와 0.5인 행렬의 determinant와 condition number를 예측한다.

### 작은 유사 사례와 계약

`matrix_diagnostics`는 유한한 정사각행렬에서 `(determinant, condition_number)`를 반환한다. `linear_forward`의 weight는 `(out_features, in_features)`이고 입력은 `(batch, in_features)`, bias는 `(out_features,)`다. shape이 맞지 않으면 `ValueError`다.

### 구현

`matrix_diagnostics`와 `linear_forward`를 구현한다.

<details>
<summary>힌트 1: 관찰할 상태와 개념</summary>

determinant는 역행렬 존재 여부의 단서이고, condition number는 축별 스케일 불균형과 민감도를 보여주는 별도 값이다.
</details>

<details>
<summary>힌트 2: 작은 trace 또는 Shape 흐름</summary>

PyTorch-style weight `(out, in)`은 batch 입력 `(batch, in)`과 바로 곱할 수 없다. 어느 축을 맞춰야 `(batch, out)`이 되는지 종이에 먼저 적는다.
</details>


In [7]:
# TODO: E02
def matrix_diagnostics(matrix: Array) -> tuple[float, float]:
    """Return (determinant, condition number) for a finite square matrix."""
    matrix = np.asarray(matrix)

    if matrix.ndim != 2 or matrix.shape[0] != matrix.shape[1]:
        raise ValueError("matrix must be square")

    if not np.all(np.isfinite(matrix)):
        raise ValueError("matrix must contain only finite values")

    det = np.linalg.det(matrix)
    cond = np.linalg.cond(matrix)

    return det, cond


def linear_forward(inputs: Array, weight: Array, bias: Array) -> Array:
    """Apply a batch linear layer with PyTorch-style weight shape (out_features, in_features)."""
    inputs = np.asarray(inputs)
    weight = np.asarray(weight)
    bias = np.asarray(bias)

    weight = weight.T

    return inputs @ weight + bias


In [8]:
# provided-fixture: E02
# 먼저 E02 구현 셀을 실행하세요.

diagnostic_matrix = np.array([[2.0, 0.0], [0.0, 0.5]])
inputs = np.array([[1.0, 2.0, 3.0], [0.0, 1.0, 2.0]])
weight = np.array([[1.0, 0.0, -1.0], [0.0, 2.0, 1.0]])
bias = np.array([0.5, -1.0])

determinant, condition = matrix_diagnostics(diagnostic_matrix)
output = linear_forward(inputs, weight, bias)

print("determinant:", determinant)
print("condition number:", condition)
print("inputs shape:", inputs.shape)
print("weight shape (out, in):", weight.shape)
print("bias shape:", bias.shape)
print("output shape:", output.shape)
print("output:\n", output)


determinant: 1.0
condition number: 4.0
inputs shape: (2, 3)
weight shape (out, in): (2, 3)
bias shape: (2,)
output shape: (2, 2)
output:
 [[-1.5  6. ]
 [-1.5  3. ]]


### 테스트와 실패 진단

아래 `check_e02()`로 정상 batch, batch 크기 1, 비정사각 행렬과 폭이 맞지 않는 입력을 확인한다. 출력이 전치되어 보이면 weight convention과 입력 shape을 먼저 출력해 본다.


In [9]:
# test-check: E02
def check_e02() -> None:
    # normal
    determinant, condition = matrix_diagnostics(
        np.array([[2.0, 0.0], [0.0, 0.5]])
    )
    np.testing.assert_allclose([determinant, condition], [1.0, 4.0])
    inputs = np.array([[1.0, 2.0, 3.0], [0.0, 1.0, 2.0]])
    weight = np.array([[1.0, 0.0, -1.0], [0.0, 2.0, 1.0]])
    bias = np.array([0.5, -1.0])
    np.testing.assert_allclose(
        linear_forward(inputs, weight, bias),
        np.array([[-1.5, 6.0], [-1.5, 3.0]]),
    )

    # edge
    identity_determinant, identity_condition = matrix_diagnostics(np.eye(2))
    np.testing.assert_allclose(
        [identity_determinant, identity_condition],
        [1.0, 1.0],
    )
    output = linear_forward(np.array([[2.0, 1.0]]), np.eye(2), np.zeros(2))
    np.testing.assert_equal(output.shape, (1, 2))

    # failure
    with np.testing.assert_raises(ValueError):
        matrix_diagnostics(np.ones((2, 3)))
    with np.testing.assert_raises(ValueError):
        matrix_diagnostics(np.array([[1.0, np.nan], [0.0, 1.0]]))
    with np.testing.assert_raises(ValueError):
        linear_forward(np.ones((2, 3)), np.ones((2, 2)), np.zeros(2))

    print("E02 checks passed")


check_e02()


E02 checks passed


### 결과 해석

determinant가 0이 아닌 것과 condition number가 작은 것은 다른 주장임을 설명하고, batch 축이 결과에 그대로 남는 이유를 설명한다.


## E03. 두 affine 레이어를 한 단계로 합치기

### 실제 사용 맥락
활성화 함수가 없는 두 개의 작은 dense layer를 배포 전에 하나의 affine 레이어로 바꿔도 되는지 검증한다.

### 실행 전 회상·예측

- 첫 레이어와 두 번째 레이어 중 실제로 어느 것이 먼저 적용되는지 적는다.
- 입력이 영벡터일 때 합성된 레이어의 출력이 무엇과 같을지 적는다.

### 작은 유사 사례와 계약

`column_weighted_sum`은 열벡터 행렬과 같은 길이의 계수를 받아 그 열벡터들의 가중합을 반환한다. `compose_affine`은 first를 적용한 뒤 second를 적용한 것과 동등한 `(weight, bias)`를 반환한다. 두 레이어의 중간 feature 수가 다르거나 bias 길이가 맞지 않으면 `ValueError`다.

### 구현

`column_weighted_sum`과 `compose_affine`을 구현하고 E02의 `linear_forward`로 순차 적용과 합성 적용을 비교한다.

<details>
<summary>힌트 1: 관찰할 상태와 개념</summary>

row-vector batch convention에서는 `first`의 출력 feature 수가 `second`의 입력 feature 수와 같아야 한다. 영입력을 넣어 보면 합성 bias를 따로 검증할 수 있다.
</details>

<details>
<summary>힌트 2: 작은 trace 또는 Shape 흐름</summary>

입력 `(batch, in)`을 first에 넣어 `(batch, hidden)`을 만든 뒤 second에 넣어 `(batch, out)`을 만든다. 이 두 단계의 결과와 한 단계의 결과가 모든 batch 행에서 같아야 한다.
</details>


In [10]:
# TODO: E03
def column_weighted_sum(columns: Array, coefficients: Array) -> Array:
    """Return the vector represented by coefficients in a matrix's column-vector basis."""
    columns = np.asarray(columns)
    coefficients = np.asarray(coefficients)

    return columns @ coefficients


def compose_affine(
    first_weight: Array,
    first_bias: Array,
    second_weight: Array,
    second_bias: Array,
) -> tuple[Array, Array]:
    first_weight = np.asarray(first_weight)
    second_weight = np.asarray(second_weight)
    first_bias = np.asarray(first_bias)
    second_bias = np.asarray(second_bias)

    return second_weight @ first_weight, first_bias @ second_weight.T + second_bias


In [11]:
# provided-fixture: E03
# 먼저 E02와 E03 구현 셀을 실행하세요.

first_weight = np.array([[1.0, 2.0], [0.0, 1.0]])
first_bias = np.array([1.0, -1.0])
second_weight = np.array([[2.0, 0.0], [1.0, 1.0]])
second_bias = np.array([0.0, 3.0])
inputs = np.array([
    [2.0, -1.0],
    [0.0, 0.0],
])
columns = np.array([[1.0, 1.0], [1.0, -1.0]])
coefficients = np.array([4.0, 5.0])

combined_weight, combined_bias = compose_affine(
    first_weight,
    first_bias,
    second_weight,
    second_bias,
)
sequential = linear_forward(
    linear_forward(inputs, first_weight, first_bias),
    second_weight,
    second_bias,
)
combined = linear_forward(inputs, combined_weight, combined_bias)
weighted_sum = column_weighted_sum(columns, coefficients)

print("combined weight:\n", combined_weight)
print("combined bias:", combined_bias)
print("sequential output:\n", sequential)
print("combined output:\n", combined)
print("zero-input output:", combined[1])
print("column weighted sum:", weighted_sum)


combined weight:
 [[2. 4.]
 [1. 3.]]
combined bias: [2. 3.]
sequential output:
 [[2. 2.]
 [2. 3.]]
combined output:
 [[2. 2.]
 [2. 3.]]
zero-input output: [2. 3.]
column weighted sum: [ 9. -1.]


### 테스트와 실패 진단

아래 `check_e03()`으로 순차 결과와 합성 결과, 영입력의 bias, 열벡터 가중합, 잘못된 shape을 확인한다. 불일치하면 weight와 bias 중 어느 단계에서 적용 순서가 어긋났는지 기록한다.


In [12]:
# test-check: E03
def check_e03() -> None:
    # normal
    first_weight = np.array([[1.0, 2.0], [0.0, 1.0]])
    first_bias = np.array([1.0, -1.0])
    second_weight = np.array([[2.0, 0.0], [1.0, 1.0]])
    second_bias = np.array([0.0, 3.0])
    combined_weight, combined_bias = compose_affine(
        first_weight,
        first_bias,
        second_weight,
        second_bias,
    )
    inputs = np.array([[2.0, -1.0], [0.0, 0.0]])
    sequential = linear_forward(
        linear_forward(inputs, first_weight, first_bias),
        second_weight,
        second_bias,
    )
    np.testing.assert_allclose(
        linear_forward(inputs, combined_weight, combined_bias),
        sequential,
    )
    np.testing.assert_allclose(
        column_weighted_sum(
            np.array([[1.0, 1.0], [1.0, -1.0]]),
            np.array([4.0, 5.0]),
        ),
        np.array([9.0, -1.0]),
    )

    # edge
    np.testing.assert_allclose(
        linear_forward(np.zeros((1, 2)), combined_weight, combined_bias)[0],
        combined_bias,
    )

    # failure
    with np.testing.assert_raises(ValueError):
        compose_affine(np.eye(2), np.zeros(2), np.ones((3, 4)), np.zeros(3))
    with np.testing.assert_raises(ValueError):
        compose_affine(np.eye(2), np.zeros(3), np.eye(2), np.zeros(2))
    with np.testing.assert_raises(ValueError):
        column_weighted_sum(np.ones((2, 3)), np.ones(2))

    print("E03 checks passed")


check_e03()


E03 checks passed


### 결과 해석

영입력 결과가 일반적으로 0이 아니라는 점으로 선형변환과 affine 변환을 구분하고, activation이 없을 때만 이 합성이 가능한 이유를 설명한다.


## E04. 해의 종류와 최소제곱 투영

### 실제 사용 맥락
작은 회귀 전처리 도구가 정확한 선형시스템인지, 자유변수가 남는지, 모순인지 먼저 분류하고, 정확해가 없을 때는 가장 가까운 투영을 반환해야 한다.

### 실행 전 회상·예측

- `x1 + x2 = 5`처럼 독립 제약이 부족한 경우의 해 종류를 적는다.
- 최소제곱 residual은 design의 열공간과 어떤 내적 관계여야 하는지 적는다.

### 작은 유사 사례와 계약

`classify_linear_system`은 `unique`, `infinite`, `inconsistent` 중 하나를 반환한다. `least_squares_projection`은 `(coefficients, projection, residual)`을 반환하며 `projection + residual == target`이고 residual은 design의 모든 열과 직교해야 한다. target shape이 맞지 않으면 `ValueError`다.

### 구현

두 함수를 구현하고 rank가 부족한 경우에도 최소제곱 residual의 직교성을 확인한다.

<details>
<summary>힌트 1: 관찰할 상태와 개념</summary>

정확한 해의 분류에는 계수행렬과 확대행렬의 rank가 모두 필요하다. 최소제곱에서는 exact equality 대신 열공간 위의 가장 가까운 점을 찾는다.
</details>

<details>
<summary>힌트 2: 작은 trace 또는 Shape 흐름</summary>

design shape이 `(n_samples, n_features)`이면 projection과 residual은 `(n_samples,)`다. 마지막에 `design.T @ residual`의 shape은 `(n_features,)`가 되어야 한다.
</details>


In [13]:
# TODO: E04
def classify_linear_system(coefficients: Array, targets: Array) -> str:
    """Return exactly one of: 'unique', 'infinite', or 'inconsistent'."""
    coefficients = np.asarray(coefficients)
    targets = np.asarray(targets)

    if coefficients.ndim != 2:
        raise ValueError("coefficients must be 2D")

    if targets.ndim != 1 or targets.shape[0] != coefficients.shape[0]:
        raise ValueError("targets must have one value per equation")

    rank_a = np.linalg.matrix_rank(coefficients)
    augmented = np.column_stack((coefficients, targets))
    rank_augmented = np.linalg.matrix_rank(augmented)

    if rank_augmented > rank_a:
        return "inconsistent"

    n_unknowns = coefficients.shape[1]

    if rank_a == n_unknowns:
        return "unique"

    return "infinite"


def least_squares_projection(
    design: Array, target: Array
) -> tuple[Array, Array, Array]:
    """Return (coefficients, projection, residual) for a least-squares fit of target."""
    coefficients, *_ = np.linalg.lstsq(design, target, rcond=None)
    projection = design @ coefficients
    residual = target - projection

    return coefficients, projection, residual


In [14]:
# provided-fixture: E04
# 먼저 E04 구현 셀을 실행하세요.

systems = {
    "unique": (
        np.array([[2.0, 1.0], [1.0, -1.0]]),
        np.array([5.0, 1.0]),
    ),
    "infinite": (
        np.array([[1.0, 1.0], [2.0, 2.0]]),
        np.array([5.0, 10.0]),
    ),
    "inconsistent": (
        np.array([[1.0, 1.0], [2.0, 2.0]]),
        np.array([5.0, 11.0]),
    ),
}
design = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
target = np.array([1.0, 2.0, 4.0])
rank_deficient_design = np.array(
    [[1.0, 1.0], [2.0, 2.0], [3.0, 3.0]]
)
rank_deficient_target = np.array([1.0, 2.0, 4.0])

classifications = {
    expected: classify_linear_system(coefficients, targets)
    for expected, (coefficients, targets) in systems.items()
}
coefficients, projection, residual = least_squares_projection(design, target)
_, _, rank_deficient_residual = least_squares_projection(
    rank_deficient_design,
    rank_deficient_target,
)

print("classifications:", classifications)
print("least-squares coefficients:", coefficients)
print("projection:", projection)
print("residual:", residual)
print("projection + residual:", projection + residual)
print("design.T @ residual:", design.T @ residual)
print("rank-deficient orthogonality:", rank_deficient_design.T @ rank_deficient_residual)


classifications: {'unique': 'unique', 'infinite': 'infinite', 'inconsistent': 'inconsistent'}
least-squares coefficients: [1.33333333 2.33333333]
projection: [1.33333333 2.33333333 3.66666667]
residual: [-0.33333333 -0.33333333  0.33333333]
projection + residual: [1. 2. 4.]
design.T @ residual: [-2.22044605e-16 -8.88178420e-16]
rank-deficient orthogonality: [4.4408921e-15 4.4408921e-15]


### 테스트와 실패 진단

아래 `check_e04()`로 세 해의 종류와 rank-deficient 최소제곱을 확인한다. residual 직교 검사가 실패하면 residual 방향과 `projection + residual`의 부호를 먼저 확인한다.


In [18]:
# test-check: E04
def check_e04() -> None:
    # normal
    np.testing.assert_equal(
        classify_linear_system(
            np.array([[2.0, 1.0], [1.0, -1.0]]),
            np.array([5.0, 1.0]),
        ),
        "unique",
    )
    np.testing.assert_equal(
        classify_linear_system(
            np.array([[1.0, 1.0], [2.0, 2.0]]),
            np.array([5.0, 10.0]),
        ),
        "infinite",
    )
    np.testing.assert_equal(
        classify_linear_system(
            np.array([[1.0, 1.0], [2.0, 2.0]]),
            np.array([5.0, 11.0]),
        ),
        "inconsistent",
    )
    design = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
    target = np.array([1.0, 2.0, 4.0])
    coefficients, projection, residual = least_squares_projection(design, target)
    np.testing.assert_allclose(coefficients, np.array([4.0 / 3.0, 7.0 / 3.0]))
    np.testing.assert_allclose(projection + residual, target)
    np.testing.assert_allclose(design.T @ residual, np.zeros(2), atol=1e-12)

    # edge
    rank_deficient = np.array([[1.0, 1.0], [2.0, 2.0], [3.0, 3.0]])
    _, _, rank_deficient_residual = least_squares_projection(
        rank_deficient,
        np.array([1.0, 2.0, 4.0]),
    )
    np.testing.assert_allclose(
        rank_deficient.T @ rank_deficient_residual,
        np.zeros(2),
        atol=1e-12,
    )

    # failure
    with np.testing.assert_raises(ValueError):
        classify_linear_system(np.eye(2), np.ones((2, 1)))
    with np.testing.assert_raises(ValueError):
        least_squares_projection(np.eye(2), np.ones(3))

    print("E04 checks passed")


check_e04()


E04 checks passed


### 결과 해석

무한해는 자유변수 때문이라는 점과, 최소제곱의 오차 최소화가 단순히 작은 수를 고르는 것이 아니라 수직 투영 조건이라는 점을 설명한다.


## E05. 고유벡터 좌표에서 변환 적용

### 실제 사용 맥락
어떤 선형변환이 고유벡터 방향별 스케일만 바꾼다고 알려졌을 때, 원래 좌표의 입력을 고유좌표로 바꾸고 각 계수를 조정한 뒤 되돌린다.

### 실행 전 회상·예측

- `P^-1`, 고유값별 스케일, P의 역할을 순서대로 적는다.
- P의 열이 선형독립이 아니면 어느 단계가 막히는지, 고유값이 음수면 해당 방향에서 무슨 일이 일어나는지 적는다.

### 작은 유사 사례와 계약

`apply_in_eigenbasis`는 `(eigen_coordinates, scaled_coordinates, result)`를 반환한다. eigenvectors는 정사각·가역이어야 하고 eigenvalues 길이는 열 수와 같아야 한다. 그렇지 않으면 `ValueError`다.

### 구현

`apply_in_eigenbasis`를 구현한다.

<details>
<summary>힌트 1: 관찰할 상태와 개념</summary>

P의 각 열은 한 고유벡터 방향이다. 입력을 계수로 바꾸려면 그 열들을 기준으로 한 좌표계로 먼저 옮겨야 한다.
</details>

<details>
<summary>힌트 2: 작은 trace 또는 Shape 흐름</summary>

`vector (n,)`에서 좌표 `(n,)`를 만들고, 같은 shape의 고유값과 성분별로 곱한 뒤 다시 원래 좌표 `(n,)`로 돌아온다.
</details>


In [25]:
# TODO: E05
def apply_in_eigenbasis(
    vector: Array,
    eigenvectors: Array,
    eigenvalues: Array,
) -> tuple[Array, Array, Array]:
    """Return (eigen_coordinates, scaled_coordinates, result) using P, D, and P inverse."""
    eigen_coordinates = np.linalg.solve(eigenvectors,vector)
    scaled_coordinates = eigenvalues * eigen_coordinates
    result = eigenvectors @ scaled_coordinates

    return eigen_coordinates, scaled_coordinates, result


In [26]:
# provided-fixture: E05
# 먼저 E05 구현 셀을 실행하세요.

vector = np.array([7.0, -1.0])
eigenvectors = np.array([[1.0, 1.0], [1.0, -1.0]])
eigenvalues = np.array([2.0, 3.0])
negative_vector = np.array([0.0, 1.0])
negative_eigenvectors = np.eye(2)
negative_eigenvalues = np.array([2.0, -1.0])

coordinates, scaled_coordinates, result = apply_in_eigenbasis(
    vector,
    eigenvectors,
    eigenvalues,
)
negative_coordinates, negative_scaled, negative_result = apply_in_eigenbasis(
    negative_vector,
    negative_eigenvectors,
    negative_eigenvalues,
)

print("original vector:", vector)
print("eigen coordinates:", coordinates)
print("scaled coordinates:", scaled_coordinates)
print("result in original coordinates:", result)
print("negative-case coordinates:", negative_coordinates)
print("negative-case scaled coordinates:", negative_scaled)
print("negative-case result:", negative_result)


original vector: [ 7. -1.]
eigen coordinates: [3. 4.]
scaled coordinates: [ 6. 12.]
result in original coordinates: [18. -6.]
negative-case coordinates: [0. 1.]
negative-case scaled coordinates: [ 0. -1.]
negative-case result: [ 0. -1.]


### 테스트와 실패 진단

아래 `check_e05()`로 세 반환값, 음수 고유값, 종속된 고유벡터 열과 잘못된 shape을 확인한다. 역행렬 오류가 나면 P의 rank와 열 방향을 먼저 점검한다.


In [27]:
# test-check: E05
def check_e05() -> None:
    # normal
    eigenvectors = np.array([[1.0, 1.0], [1.0, -1.0]])
    coordinates, scaled_coordinates, result = apply_in_eigenbasis(
        np.array([7.0, -1.0]),
        eigenvectors,
        np.array([2.0, 3.0]),
    )
    np.testing.assert_allclose(coordinates, np.array([3.0, 4.0]))
    np.testing.assert_allclose(scaled_coordinates, np.array([6.0, 12.0]))
    np.testing.assert_allclose(result, np.array([18.0, -6.0]))

    # edge
    _, negative_scaled, negative_result = apply_in_eigenbasis(
        np.array([0.0, 1.0]),
        np.eye(2),
        np.array([2.0, -1.0]),
    )
    np.testing.assert_allclose(negative_scaled, np.array([0.0, -1.0]))
    np.testing.assert_allclose(negative_result, np.array([0.0, -1.0]))

    # failure
    with np.testing.assert_raises(ValueError):
        apply_in_eigenbasis(
            np.array([1.0, 2.0]),
            np.array([[1.0, 2.0], [2.0, 4.0]]),
            np.array([2.0, 3.0]),
        )
    with np.testing.assert_raises(ValueError):
        apply_in_eigenbasis(np.ones(2), np.eye(2), np.ones(3))
    with np.testing.assert_raises(ValueError):
        apply_in_eigenbasis(np.ones(2), np.ones((2, 3)), np.ones(3))

    print("E05 checks passed")


check_e05()


E05 checks passed


### 결과 해석

고유값이 원래 좌표 성분이 아니라 고유벡터 방향의 계수에 적용된다는 점, 음수 고유값이 그 방향을 반대로 뒤집는다는 점, `P.T = P^-1`이 일반 규칙이 아니라 정규직교일 때의 특수 규칙임을 설명한다.


## E06. 중심화한 표본 공분산으로 PCA 투영

### 실제 사용 맥락
단위와 값 범위가 다른 feature를 조사할 때, 먼저 feature별 평균을 빼고 표본 공분산의 큰 분산 방향만 남겨 작은 feature 표현을 만든다.

### 실행 전 회상·예측

- `(N, d)` samples에서 mean, covariance, components, scores의 shape을 적는다.
- `[1, 2], [2, 4], [3, 6]`처럼 한 직선 위 데이터에서 0에 가까운 고유값이 뜻하는 것을 적는다.

### 작은 유사 사례와 계약
`sample_covariance(samples)`는 `(d, d)` 대칭 표본 공분산을 반환한다. 대각 원소는 각 feature의 분산이고 비대각 원소는 두 feature의 함께 변하는 경향이다. `pca_project(samples, k)`는 `(mean, descending_eigenvalues, components, scores)`를 반환한다. samples는 유한한 2D float 배열이고 `N >= 2`, `1 <= k <= d`여야 한다. components는 `(d, k)`, scores는 `(N, k)`다.

### 구현

`sample_covariance`와 `pca_project`를 구현한다.

<details>
<summary>힌트 1: 관찰할 상태와 개념</summary>

중심화는 각 행에서 같은 값을 빼는 것이 아니라 각 feature 열의 평균을 뺀다. 표본 공분산은 `N - 1`로 나뉜다.
</details>

<details>
<summary>힌트 2: 작은 trace 또는 Shape 흐름</summary>

centered samples는 `(N, d)`, covariance는 `(d, d)`, 선택한 components는 `(d, k)`, scores는 `(N, k)`다. 대칭 공분산의 고유값은 큰 순서로 정렬해야 한다.
</details>


In [ ]:
# TODO: E06
def pca_project(samples: Array, k: int) -> tuple[Array, Array, Array, Array]:
    """Return (mean, descending_eigenvalues, components, scores) for centered sample PCA."""
    samples = np.asarray(samples)

    if samples.shape[0] < 2:
        raise ValueError("at least two samples are required")

    if not 1 <= k <= samples.shape[1]:
        raise ValueError("k must be between 1 and the feature count")

    mean = samples.mean(axis=0)
    centered = samples - mean
    covariance = sample_covariance(samples)

    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    components = eigenvectors[:, :k] # (d, k)
    scores = centered @ components # (N, k)

    return mean, eigenvalues, components, scores


def sample_covariance(samples: Array) -> Array:
    """Return the symmetric sample covariance matrix of finite 2D samples with N >= 2."""
    samples = np.asarray(samples, dtype=float)

    if samples.ndim != 2:
        raise ValueError("sample is not 2D")

    if samples.shape[0] < 2:
        raise ValueError("at least two samples are required")

    if not np.all(np.isfinite(samples)):
        raise ValueError("samples must contain only finite values")

    n_samples = samples.shape[0]
    mean = samples.mean(axis=0)
    centered = samples - mean

    return centered.T @ centered / (n_samples - 1)


In [31]:
# provided-fixture: E06
# 먼저 E06 구현 셀을 실행하세요.

samples = np.array([[1.0, 2.0], [2.0, 4.0], [3.0, 6.0]])

covariance = sample_covariance(samples)
mean, eigenvalues, components_k1, scores_k1 = pca_project(samples, k=1)
_, eigenvalues_k2, components_k2, scores_k2 = pca_project(samples, k=2)
reconstructed_k1 = scores_k1 @ components_k1.T + mean

print("samples shape:", samples.shape)
print("mean:", mean, "shape:", mean.shape)
print("sample covariance:\n", covariance)
print("covariance shape:", covariance.shape)
print("descending eigenvalues:", eigenvalues)
print("k=1 components shape:", components_k1.shape)
print("k=1 scores shape:", scores_k1.shape)
print("k=1 reconstruction:\n", reconstructed_k1)
print("k=2 eigenvalues:", eigenvalues_k2)
print("k=2 components shape:", components_k2.shape)
print("k=2 scores shape:", scores_k2.shape)


samples shape: (3, 2)
mean: [2. 4.] shape: (2,)
sample covariance:
 [[1. 2.]
 [2. 4.]]
covariance shape: (2, 2)
descending eigenvalues: [5. 0.]
k=1 components shape: (2, 1)
k=1 scores shape: (3, 1)
k=1 reconstruction:
 [[1. 2.]
 [2. 4.]
 [3. 6.]]
k=2 eigenvalues: [5. 0.]
k=2 components shape: (2, 2)
k=2 scores shape: (3, 2)


### 테스트와 실패 진단

아래 `check_e06()`으로 표본 공분산, 고유값, `k=1/2`, 너무 적은 표본과 잘못된 k를 확인한다. component 부호가 바뀌어도 재구성이 같은지 살핀다.


In [53]:
# test-check: E06
def check_e06() -> None:
    # normal
    samples = np.array([[1.0, 2.0], [2.0, 4.0], [3.0, 6.0]])
    np.testing.assert_allclose(
        sample_covariance(samples),
        np.array([[1.0, 2.0], [2.0, 4.0]]),
    )
    mean, eigenvalues, components, scores = pca_project(samples, k=1)
    np.testing.assert_allclose(mean, np.array([2.0, 4.0]))
    np.testing.assert_allclose(eigenvalues, np.array([5.0, 0.0]), atol=1e-12)
    np.testing.assert_equal(components.shape, (2, 1))
    np.testing.assert_equal(scores.shape, (3, 1))
    np.testing.assert_allclose(scores @ components.T + mean, samples, atol=1e-12)

    # edge
    _, all_eigenvalues, all_components, all_scores = pca_project(samples, k=2)
    np.testing.assert_equal(all_eigenvalues.shape, (2,))
    np.testing.assert_equal(all_components.shape, (2, 2))
    np.testing.assert_equal(all_scores.shape, (3, 2))

    # failure
    with np.testing.assert_raises(ValueError):
        pca_project(np.array([[1.0, 2.0]]), k=1)
    with np.testing.assert_raises(ValueError):
        sample_covariance(np.array([[1.0, 2.0]]))
    with np.testing.assert_raises(ValueError):
        pca_project(np.array([[1.0, 2.0], [2.0, 4.0]]), k=0)
    with np.testing.assert_raises(ValueError):
        pca_project(np.array([[1.0, 2.0], [2.0, 4.0]]), k=3)
    with np.testing.assert_raises(ValueError):
        sample_covariance(np.array([[1.0, np.nan], [2.0, 4.0]]))
    with np.testing.assert_raises(ValueError):
        pca_project(np.array([[1.0, np.nan], [2.0, 4.0]]), k=1)

    print("E06 checks passed")


check_e06()


E06 checks passed


### 결과 해석

대각 원소가 각 feature의 퍼짐이고 비대각 원소가 함께 변하는 경향이라는 점, 0 고유값 방향이 표본에 분산이 없다는 뜻, 표준화가 이 구현의 필수 규칙이 아니라 단위·스케일에 따라 선택하는 전처리라는 점을 적는다.
